# Weak supervision & programmatic labeling — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Combining noisy labeling functions
Weak supervision replaces hand labels with labeling functions whose coverage and conflicts must be measured. These examples inspect coverage, conflicts, majority vote, weighted vote, and abstention.

In [ ]:
# 1) Coverage tells how often each labeling function speaks
L=np.array([[1,1,0],[1,-1,1],[-1,0,0],[0,0,0],[1,1,1]]) # -1 means abstain
coverage=(L!=-1).mean(axis=0)
plt.figure(figsize=(6,3)); plt.bar(['LF1','LF2','LF3'],coverage); plt.ylim(0,1); plt.title('coverage by labeling function')
assert np.allclose(coverage,[0.8,0.8,1.0])

In [ ]:
# 2) Conflict rate finds rows where functions disagree
L=np.array([[1,1,0],[1,-1,1],[-1,0,0],[0,0,0],[1,1,1]]); conflict=np.array([len(set(r[r!=-1]))>1 for r in L])
plt.figure(figsize=(6,3)); plt.bar(range(5),conflict.astype(int)); plt.ylim(0,1.1); plt.title('conflicting weak labels')
assert conflict.mean()==0.2

In [ ]:
# 3) Majority vote ignores abstentions and resolves weak labels
L=np.array([[1,1,0],[1,-1,1],[-1,0,0],[0,0,0],[1,1,1]]); sums=np.where(L==-1,0,L).sum(axis=1); mv=(sums>=1).astype(int)
plt.figure(figsize=(6,3)); plt.bar(range(5),sums); plt.axhline(.5,c='r',ls='--'); plt.title('vote sums after abstentions are removed')
assert np.array_equal(mv,np.array([1,1,0,0,1]))

In [ ]:
# 4) Accuracy weights can overturn a noisy conflict
row=np.array([1,1,0]); w=np.array([.9,.8,.4]); score=(row*w).sum()/w.sum()
plt.figure(figsize=(6,3)); plt.bar(['LF1','LF2','LF3'],row*w); plt.title(f'weighted positive score = {score:.3f}')
assert abs(score-0.8095238095)<1e-9 and score>.5

In [ ]:
# 5) Abstention trades coverage for precision in a thresholded label model
conf=np.array([.95,.7,.55,.45,.2]); yhat=np.array([1,1,1,0,0]); thresholds=np.linspace(.5,.9,30); cov=[(conf>=t).mean() for t in thresholds]
plt.figure(figsize=(6,3)); plt.plot(thresholds,cov); plt.xlabel('confidence threshold'); plt.ylabel('labeled fraction'); plt.title('higher threshold abstains more')
assert cov[0]==0.6 and cov[-1]==0.2